# Debug Notebook: Scheduler Rules Benchmark (Simulation)

This notebook benchmarks scheduler-rule optimization strategies for speculative decoding traffic:
- separate short vs long lanes
- avoid mixing tiny and long requests in same batch
- batch by similar remaining decode length
- continuous batching with a max queue-wait bound

It produces paired policy deltas, robust statistics (median/p90), and bootstrap CI outputs.

In [ ]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except ModuleNotFoundError:
    torch = None

search_starts = []
if '__file__' in globals():
    search_starts.append(Path(__file__).resolve().parent)
search_starts.append(Path.cwd().resolve())

project_root = None
seen = set()
for start in search_starts:
    for p in [start, *start.parents]:
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'speculative.py').exists():
            project_root = p
            break
    if project_root is not None:
        break

if project_root is None:
    raise RuntimeError('Could not find project root containing src/speculative.py')

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

os.environ.setdefault('SPECDEC_HF_OFFLINE_FIRST', '1')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

from config import DRAFT_MODELS, DRAFT_QUANT, TARGET_MODEL_ID, TARGET_QUANT, REGIMES
from speculative import load_model_on_device, speculative_decode_sample
from utils import set_seed

results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
manifests_dir = project_root / 'manifests'

print(f'Project root: {project_root}')
print(f'CUDA available: {bool(torch is not None and torch.cuda.is_available())}')

In [ ]:
# Profiling + simulation controls
DEVICE = 'cuda:0'
DRAFT_LABEL = '0.5B'
REGIME_NAME = 'deterministic'
K = 8

PROFILE_MAX_NEW_TOKENS = [16, 64, 160]
N_PROMPTS_PER_TASK = 6
PROFILE_REPEATS = 1
USE_EXISTING_PROFILE_IF_AVAILABLE = True

SHORT_MAX_TOKENS = 32
MEDIUM_MAX_TOKENS = 96

ARRIVAL_RATES_RPS = [0.20, 0.35, 0.50]
NUM_REQUESTS_PER_TRIAL = 300
NUM_TRIALS = 25
SIM_SEED = 2026

MAX_BATCH_SIZE = 8
BATCH_WINDOW_S = 0.040
MAX_QUEUE_WAIT_S = 0.060
SIMILARITY_RATIO_BOUND = 1.5

PROFILE_CSV = results_dir / 'gpu_scheduler_service_profile.csv'
RUNS_CSV = results_dir / 'gpu_scheduler_simulation_runs.csv'
DELTA_CSV = results_dir / 'gpu_scheduler_policy_delta.csv'
ROBUST_CSV = results_dir / 'gpu_scheduler_policy_robust_delta.csv'
BOOTSTRAP_CSV = results_dir / 'gpu_scheduler_policy_bootstrap_ci.csv'

print('Scheduler benchmark config ready.')

In [ ]:
def token_bucket(max_new_tokens: int) -> str:
    if max_new_tokens <= SHORT_MAX_TOKENS:
        return 'short'
    if max_new_tokens <= MEDIUM_MAX_TOKENS:
        return 'medium'
    return 'long'

def load_prompt_rows(n_per_task: int = 6) -> list[dict]:
    rows = []
    for task in ['gsm8k', 'mmlu', 'cnndm']:
        path = manifests_dir / f'{task}_data.json'
        if not path.exists():
            continue
        with open(path, 'r') as f:
            data = json.load(f)
        for item in data[:n_per_task]:
            prompt = item.get('prompt')
            if prompt:
                rows.append({
                    'task': task,
                    'sample_id': item.get('sample_id', ''),
                    'prompt': prompt,
                })
    if not rows:
        raise RuntimeError('No prompts found in manifests/*_data.json')
    return rows

prompt_rows = load_prompt_rows(N_PROMPTS_PER_TASK)
print(f'Loaded prompts: {len(prompt_rows)}')
pd.DataFrame(prompt_rows)[['task', 'sample_id']].head(10)

In [ ]:
def profile_service_times() -> pd.DataFrame:
    if USE_EXISTING_PROFILE_IF_AVAILABLE and PROFILE_CSV.exists():
        print(f'Using existing profile: {PROFILE_CSV}')
        return pd.read_csv(PROFILE_CSV)

    if torch is None or not torch.cuda.is_available():
        raise RuntimeError('CUDA is required to generate new service profile. Set USE_EXISTING_PROFILE_IF_AVAILABLE=True with an existing CSV.')

    target_model, target_tokenizer = load_model_on_device(
        TARGET_MODEL_ID,
        device=DEVICE,
        quant_mode=TARGET_QUANT,
    )
    draft_model, _ = load_model_on_device(
        DRAFT_MODELS[DRAFT_LABEL],
        device=DEVICE,
        quant_mode=DRAFT_QUANT,
    )

    regime = REGIMES[REGIME_NAME]
    rows = []
    for rep in range(PROFILE_REPEATS):
        for i, p in enumerate(prompt_rows):
            for mnt in PROFILE_MAX_NEW_TOKENS:
                seed = SIM_SEED + rep * 100000 + i * 1000 + mnt
                set_seed(seed)
                out = speculative_decode_sample(
                    target_model,
                    draft_model,
                    target_tokenizer,
                    p['prompt'],
                    max_new_tokens=mnt,
                    k=K,
                    temperature=regime.temperature,
                    top_p=regime.top_p,
                    return_timing_breakdown=True,
                )
                rows.append({
                    'task': p['task'],
                    'sample_id': p['sample_id'],
                    'repeat': rep,
                    'max_new_tokens': mnt,
                    'bucket': token_bucket(mnt),
                    'latency_s': float(out.get('latency_s', 0.0)),
                    'draft_elapsed_s': float(out.get('draft_elapsed_s', 0.0)),
                    'verify_elapsed_s': float(out.get('verify_elapsed_s', 0.0)),
                    'ttft_ms': float(out.get('ttft_ms', 0.0)),
                    'num_tokens': int(out.get('num_tokens', 0)),
                })

    df = pd.DataFrame(rows)
    df.to_csv(PROFILE_CSV, index=False)
    print(f'Saved profile: {PROFILE_CSV}')
    return df

service_profile = profile_service_times()
service_profile.head()

In [ ]:
bucket_stats = service_profile.groupby('bucket', as_index=False).agg(
    mean_latency_s=('latency_s', 'mean'),
    p50_latency_s=('latency_s', 'median'),
    p90_latency_s=('latency_s', lambda x: float(np.quantile(x, 0.90))),
    n=('latency_s', 'count'),
)

latency_samples_by_bucket = {
    b: g['latency_s'].to_numpy()
    for b, g in service_profile.groupby('bucket')
}

if not latency_samples_by_bucket:
    raise RuntimeError('Service profile is empty.')

display(bucket_stats)

In [ ]:
POLICIES = [
    {
        'name': 'fifo_mixed_window',
        'split_lanes': False,
        'similarity_pick': False,
        'continuous': False,
    },
    {
        'name': 'lane_split_window',
        'split_lanes': True,
        'similarity_pick': False,
        'continuous': False,
    },
    {
        'name': 'length_similarity_window',
        'split_lanes': False,
        'similarity_pick': True,
        'continuous': False,
    },
    {
        'name': 'full_rules_continuous',
        'split_lanes': True,
        'similarity_pick': True,
        'continuous': True,
    },
]

POLICY_BASELINE = 'fifo_mixed_window'

def sample_bucket(rng: np.random.Generator) -> str:
    # Use empirical bucket mix from profile.
    counts = service_profile['bucket'].value_counts(normalize=True)
    buckets = counts.index.tolist()
    probs = counts.values
    return str(rng.choice(buckets, p=probs))

def sample_service_for_bucket(bucket: str, rng: np.random.Generator) -> float:
    arr = latency_samples_by_bucket[bucket]
    return float(rng.choice(arr))

def bucket_length_hint(bucket: str, rng: np.random.Generator) -> int:
    if bucket == 'short':
        return int(rng.integers(8, SHORT_MAX_TOKENS + 1))
    if bucket == 'medium':
        return int(rng.integers(SHORT_MAX_TOKENS + 1, MEDIUM_MAX_TOKENS + 1))
    return int(rng.integers(MEDIUM_MAX_TOKENS + 1, max(PROFILE_MAX_NEW_TOKENS) + 1))

def generate_requests(arrival_rate_rps: float, n: int, seed: int) -> list[dict]:
    rng = np.random.default_rng(seed)
    arrivals = np.cumsum(rng.exponential(scale=1.0 / arrival_rate_rps, size=n))
    reqs = []
    for i in range(n):
        bucket = sample_bucket(rng)
        reqs.append({
            'req_id': i,
            'arrival_s': float(arrivals[i]),
            'bucket': bucket,
            'remaining_tokens': bucket_length_hint(bucket, rng),
            'service_base_s': sample_service_for_bucket(bucket, rng),
        })
    return reqs

def effective_batch_service(batch: list[dict]) -> tuple[float, float]:
    # A simple throughput model: batching helps, heterogeneity hurts.
    base_services = np.array([r['service_base_s'] for r in batch], dtype=float)
    lengths = np.array([max(r['remaining_tokens'], 1) for r in batch], dtype=float)

    b = len(batch)
    size_speedup = min(1.40, 1.0 + 0.08 * (b - 1))
    hetero_penalty = 1.0 + 0.20 * float(np.std(lengths) / max(np.mean(lengths), 1e-9))
    batch_service = float(base_services.max() / size_speedup * hetero_penalty)
    return batch_service, float(hetero_penalty)

def pick_batch(pending: list[dict], cfg: dict) -> list[dict]:
    if not pending:
        return []

    if cfg['split_lanes']:
        short_lane = [r for r in pending if r['bucket'] in ('short', 'medium')]
        long_lane = [r for r in pending if r['bucket'] == 'long']
        if short_lane and long_lane:
            # Choose lane with older head request.
            use_short = short_lane[0]['arrival_s'] <= long_lane[0]['arrival_s']
            pending_view = short_lane if use_short else long_lane
        else:
            pending_view = short_lane if short_lane else long_lane
    else:
        pending_view = pending

    if cfg['similarity_pick'] and len(pending_view) > 1:
        anchor = pending_view[0]
        a_len = max(anchor['remaining_tokens'], 1)
        scored = []
        for r in pending_view:
            ratio = max(r['remaining_tokens'], 1) / a_len
            ratio = max(ratio, 1.0 / ratio)
            scored.append((ratio, abs(r['remaining_tokens'] - a_len), r))
        scored.sort(key=lambda x: (x[0], x[1], x[2]['arrival_s']))
        chosen = [t[2] for t in scored if t[0] <= SIMILARITY_RATIO_BOUND][:MAX_BATCH_SIZE]
        if not chosen:
            chosen = pending_view[:MAX_BATCH_SIZE]
        return chosen

    return pending_view[:MAX_BATCH_SIZE]

def simulate_policy(requests: list[dict], cfg: dict) -> pd.DataFrame:
    pending = []
    arrivals = sorted(requests, key=lambda r: r['arrival_s'])
    i = 0
    t = 0.0
    done = []

    while i < len(arrivals) or pending:
        if not pending and i < len(arrivals):
            t = max(t, arrivals[i]['arrival_s'])

        # Pull all arrivals up to current time.
        while i < len(arrivals) and arrivals[i]['arrival_s'] <= t:
            pending.append(arrivals[i])
            i += 1

        if not pending:
            continue

        pending.sort(key=lambda r: r['arrival_s'])
        oldest_wait = t - pending[0]['arrival_s']

        if cfg['continuous']:
            # In continuous mode, dispatch immediately when oldest request hits wait bound,
            # otherwise keep filling until next arrival or batch full.
            if len(pending) < MAX_BATCH_SIZE and oldest_wait < MAX_QUEUE_WAIT_S and i < len(arrivals):
                next_arrival = arrivals[i]['arrival_s']
                if next_arrival <= t + BATCH_WINDOW_S and next_arrival - pending[0]['arrival_s'] < MAX_QUEUE_WAIT_S:
                    t = next_arrival
                    continue
        else:
            # Windowed batching: wait a fixed window if possible before dispatch.
            if len(pending) < MAX_BATCH_SIZE and i < len(arrivals) and arrivals[i]['arrival_s'] <= t + BATCH_WINDOW_S:
                t = arrivals[i]['arrival_s']
                continue

        batch = pick_batch(pending, cfg)
        if not batch:
            t = arrivals[i]['arrival_s'] if i < len(arrivals) else t
            continue

        batch_ids = {r['req_id'] for r in batch}
        pending = [r for r in pending if r['req_id'] not in batch_ids]

        start_s = t
        batch_service_s, hetero_penalty = effective_batch_service(batch)
        finish_s = start_s + batch_service_s

        for r in batch:
            wait_s = max(0.0, start_s - r['arrival_s'])
            latency_s = finish_s - r['arrival_s']
            done.append({
                'req_id': r['req_id'],
                'bucket': r['bucket'],
                'arrival_s': r['arrival_s'],
                'remaining_tokens': r['remaining_tokens'],
                'service_base_s': r['service_base_s'],
                'wait_s': wait_s,
                'service_s': batch_service_s,
                'latency_s': latency_s,
                'batch_size': len(batch),
                'hetero_penalty': hetero_penalty,
            })

        t = finish_s

    return pd.DataFrame(done)

print('Simulation functions ready.')

In [ ]:
sim_rows = []
for rate in ARRIVAL_RATES_RPS:
    for trial in range(NUM_TRIALS):
        req_seed = SIM_SEED + int(rate * 10000) + trial
        reqs = generate_requests(rate, NUM_REQUESTS_PER_TRIAL, req_seed)

        for cfg in POLICIES:
            per_req = simulate_policy(reqs, cfg)
            sim_rows.append({
                'arrival_rate_rps': rate,
                'trial': trial,
                'policy': cfg['name'],
                'n_requests': int(len(per_req)),
                'mean_latency_s': float(per_req['latency_s'].mean()),
                'p50_latency_s': float(np.quantile(per_req['latency_s'], 0.50)),
                'p90_latency_s': float(np.quantile(per_req['latency_s'], 0.90)),
                'mean_wait_s': float(per_req['wait_s'].mean()),
                'p90_wait_s': float(np.quantile(per_req['wait_s'], 0.90)),
                'mean_batch_size': float(per_req['batch_size'].mean()),
                'mean_hetero_penalty': float(per_req['hetero_penalty'].mean()),
                'throughput_rps': float(len(per_req) / (per_req['latency_s'].max() - per_req['arrival_s'].min())),
            })

sim_runs = pd.DataFrame(sim_rows)
sim_runs.to_csv(RUNS_CSV, index=False)
print(f'Saved simulation runs: {RUNS_CSV}')
sim_runs.head()

In [ ]:
METRICS = ['mean_latency_s', 'p90_latency_s', 'mean_wait_s', 'p90_wait_s', 'throughput_rps']
KEYS = ['arrival_rate_rps', 'trial']

base = sim_runs[sim_runs['policy'] == POLICY_BASELINE][KEYS + METRICS].rename(columns={m: f'{m}_base' for m in METRICS})

paired_rows = []
for p in sorted(sim_runs['policy'].unique()):
    if p == POLICY_BASELINE:
        continue
    cmp_df = sim_runs[sim_runs['policy'] == p][KEYS + METRICS].rename(columns={m: f'{m}_cmp' for m in METRICS})
    merged = base.merge(cmp_df, on=KEYS, how='inner')
    if merged.empty:
        continue

    for m in METRICS:
        delta = merged[f'{m}_cmp'] - merged[f'{m}_base']
        merged[f'{m}_delta'] = delta
        merged[f'{m}_delta_pct'] = np.where(merged[f'{m}_base'] != 0, delta / merged[f'{m}_base'] * 100.0, np.nan)

    merged['baseline_policy'] = POLICY_BASELINE
    merged['compare_policy'] = p
    paired_rows.append(merged)

paired_df = pd.concat(paired_rows, ignore_index=True) if paired_rows else pd.DataFrame()

mean_rows = []
for p in sorted(paired_df['compare_policy'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_policy'] == p]
    for m in METRICS:
        b = float(block[f'{m}_base'].mean())
        c = float(block[f'{m}_cmp'].mean())
        d = float(block[f'{m}_delta'].mean())
        pct = (d / b * 100.0) if b != 0 else np.nan
        # For throughput, higher is better; invert sign for improvement metric.
        improve = pct if m == 'throughput_rps' else -pct
        mean_rows.append({
            'baseline_policy': POLICY_BASELINE,
            'compare_policy': p,
            'metric': m,
            'baseline_mean': b,
            'compare_mean': c,
            'abs_delta_compare_minus_base': d,
            'pct_delta_compare_minus_base': pct,
            'improvement_pct': improve,
            'n_pairs': int(len(block)),
        })

policy_delta = pd.DataFrame(mean_rows)
policy_delta.to_csv(DELTA_CSV, index=False)
display(policy_delta)

In [ ]:
robust_rows = []
for p in sorted(paired_df['compare_policy'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_policy'] == p]
    for m in METRICS:
        d = block[f'{m}_delta'].to_numpy()
        robust_rows.append({
            'baseline_policy': POLICY_BASELINE,
            'compare_policy': p,
            'metric': m,
            'median_delta': float(np.median(d)),
            'p90_delta': float(np.quantile(d, 0.90)),
            'p10_delta': float(np.quantile(d, 0.10)),
            'median_abs_delta': float(np.median(np.abs(d))),
            'p90_abs_delta': float(np.quantile(np.abs(d), 0.90)),
            'n_pairs': int(len(d)),
        })

robust_delta = pd.DataFrame(robust_rows)
robust_delta.to_csv(ROBUST_CSV, index=False)
display(robust_delta)

In [ ]:
BOOTSTRAP_SAMPLES = 20000
BOOTSTRAP_SEED = 42
rng = np.random.default_rng(BOOTSTRAP_SEED)

ci_rows = []
for p in sorted(paired_df['compare_policy'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_policy'] == p]
    n = len(block)
    if n == 0:
        continue
    for m in METRICS:
        d = block[f'{m}_delta'].to_numpy()
        means = np.empty(BOOTSTRAP_SAMPLES, dtype=float)
        for i in range(BOOTSTRAP_SAMPLES):
            idx = rng.integers(0, n, n)
            means[i] = float(d[idx].mean())
        lo, hi = np.quantile(means, [0.025, 0.975])
        ci_rows.append({
            'baseline_policy': POLICY_BASELINE,
            'compare_policy': p,
            'metric': m,
            'mean_delta': float(d.mean()),
            'ci95_low': float(lo),
            'ci95_high': float(hi),
            'ci_excludes_zero': bool((lo > 0) or (hi < 0)),
            'n_pairs': int(n),
            'bootstrap_samples': int(BOOTSTRAP_SAMPLES),
        })

bootstrap_ci = pd.DataFrame(ci_rows)
bootstrap_ci.to_csv(BOOTSTRAP_CSV, index=False)
display(bootstrap_ci)

In [ ]:
# Rate-wise summary and best policy pick by p90 latency.
rate_view = sim_runs.pivot_table(
    index='arrival_rate_rps',
    columns='policy',
    values='p90_latency_s',
    aggfunc='mean',
)
display(rate_view)

best_rows = []
for rate, g in sim_runs.groupby('arrival_rate_rps'):
    ranked = g.groupby('policy', as_index=False)['p90_latency_s'].mean().sort_values('p90_latency_s')
    best_rows.append({
        'arrival_rate_rps': float(rate),
        'best_policy_by_p90_latency': str(ranked.iloc[0]['policy']),
        'best_p90_latency_s': float(ranked.iloc[0]['p90_latency_s']),
    })

best_policy_table = pd.DataFrame(best_rows)
display(best_policy_table)

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
sim_runs.to_csv(results_dir / f'gpu_scheduler_simulation_runs_{stamp}.csv', index=False)
policy_delta.to_csv(results_dir / f'gpu_scheduler_policy_delta_{stamp}.csv', index=False)
bootstrap_ci.to_csv(results_dir / f'gpu_scheduler_policy_bootstrap_ci_{stamp}.csv', index=False)

print('Saved scheduler simulation outputs:')
print(f'  {RUNS_CSV}')
print(f'  {DELTA_CSV}')
print(f'  {ROBUST_CSV}')
print(f'  {BOOTSTRAP_CSV}')